# Exp 03: Lecture augmentations

Card: [exp03_lecture_augmentations.md](../../experiments/exp03_lecture_augmentations.md)


In [ ]:
import os

import albumentations as A
import matplotlib.pyplot as plt
import mlflow
import numpy as np
import pandas as pd
from PIL import Image

from core.config import MLFLOW_URI, ROOT, SEED, TRAIN_DIR
from core.cv import VIP, Config, CrossValidator, CVResult
from core.data import read_train
from core.gradcam import GradCamFigures

SMOKE = os.environ.get("SMOKE") == "1"
if SMOKE:
    OVERRIDES = {
        "experiment": "cnn-model-lecture-augmentations-smoke",
        "epochs": 1,
        "img_size": 64,
        "n_other": 20,
        "num_workers": 0,
        "retrain": True,
        "log_every_n_steps": 1,
    }
    N_SHOW = 2  # images per Grad-CAM figure
else:
    OVERRIDES = {}
    N_SHOW = 4
CONFIG_DIR = ROOT / "sections" / "model" / "configs"
CONFIG = Config.from_toml(
    CONFIG_DIR / "exp03_lecture_augmentations" / "resnet18.toml", **OVERRIDES
)
CURRENT = Config.from_toml(
    CONFIG_DIR / "exp01_lecture_models" / "resnet18.toml"
)  # trained in exp01, only read
FIGURES = ROOT / "sections" / "model" / "figures"
SAVE_FIGURES = not SMOKE

# dataviz reference palette, light mode; categorical slots in fixed order
MUTED = "#898781"
SERIES = {"current (exp01)": "#eb6834", "lecture": "#1baf7a"}
plt.rcParams.update(
    {
        "figure.facecolor": "#fcfcfb",
        "axes.facecolor": "#fcfcfb",
        "savefig.facecolor": "#fcfcfb",
        "axes.spines.top": False,
        "axes.spines.right": False,
        "axes.edgecolor": "#52514e",
        "xtick.color": "#52514e",
        "ytick.color": "#52514e",
        "axes.titlesize": "medium",
        "axes.titlecolor": "#52514e",
        "axes.grid": True,
        "grid.color": "#e1e0d9",
        "legend.frameon": False,
        "legend.fontsize": "small",
    }
)

mlflow.set_tracking_uri(MLFLOW_URI)
client = mlflow.MlflowClient()

## Lecture augmentations


In [ ]:
# lecture demo set, same parameters, p = 1
DEMO = {
    "Original": A.NoOp(),
    "Horizontal Flip": A.HorizontalFlip(p=1),
    "Random Brightness/Contrast": A.RandomBrightnessContrast(
        p=1, brightness_limit=0.3, contrast_limit=0.3
    ),
    "Hue/Saturation Shift": A.HueSaturationValue(
        p=1, hue_shift_limit=20, sat_shift_limit=30, val_shift_limit=20
    ),
    "Random Crop": A.RandomCrop(height=300, width=300, p=1),
    "Gaussian Blur": A.GaussianBlur(blur_limit=(7, 15), p=1),
    "Coarse Dropout": A.CoarseDropout(
        num_holes_range=(4, 8),
        hole_height_range=(0.05, 0.15),
        hole_width_range=(0.05, 0.15),
        fill=0,
        p=1,
    ),
    "CLAHE": A.CLAHE(clip_limit=6.0, tile_grid_size=(4, 4), p=1),
    "Elastic Transform": A.ElasticTransform(alpha=250, sigma=15, p=1),
}
train_df = read_train()
fig, axes = plt.subplots(len(VIP), len(DEMO), figsize=(2.1 * len(DEMO), 2.4 * len(VIP)))
for cls, row_axes in zip(VIP, axes):
    image_id = (
        train_df[train_df[cls] == 1].sample(1, random_state=SEED)["image_id"].item()
    )
    # pixel-sized parameters (crop 300, elastic) act on 448 px image, close to lecture demo image size
    image = np.array(
        Image.open(TRAIN_DIR / f"{image_id}.jpg").convert("RGB").resize((448, 448))
    )
    for (name, aug), ax in zip(DEMO.items(), row_axes):
        ax.imshow(aug(image=image)["image"])
        ax.set_axis_off()
        if cls == VIP[0]:
            ax.set_title(name, fontsize=9)
    row_axes[0].text(
        -0.08,
        0.5,
        cls,
        transform=row_axes[0].transAxes,
        rotation=90,
        va="center",
        ha="right",
    )
fig.tight_layout()
figures = {"exp03_lecture_augmentations_demo.png": fig}
plt.show()

## Training


In [ ]:
results = {
    "current (exp01)": CVResult.load(CURRENT),
    "lecture": CrossValidator(CONFIG).run(),
}

## Training curves


In [ ]:
def mean_curve(result, key):
    # epoch-level metrics come one per epoch, in step order
    runs = [
        client.get_metric_history(run_id, key) for run_id in result.run_ids.values()
    ]
    return np.mean(
        [[m.value for m in sorted(run, key=lambda m: m.step)] for run in runs], axis=0
    )


panels = [
    ("Loss (weighted BCE)", "train_loss_epoch", "val_loss"),
    ("mAP", "train_map", "val_map"),
]
fig, axes = plt.subplots(1, 2, figsize=(11, 4))
for ax, (title, train_key, val_key) in zip(axes, panels):
    for name, r in results.items():
        epochs = np.arange(1, r.config.epochs + 1)
        ax.plot(epochs, mean_curve(r, train_key), color=SERIES[name], ls=":")
        ax.plot(epochs, mean_curve(r, val_key), color=SERIES[name], label=name)
    ax.set(title=f"{title}, mean over {len(CONFIG.folds)} folds", xlabel="epoch")
axes[1].plot([], [], color=MUTED, label="validation")
axes[1].plot([], [], color=MUTED, ls=":", label="train")
axes[1].legend(loc="lower right", title="augmentations", alignment="left")
fig.tight_layout()
figures["exp03_lecture_augmentations_training.png"] = fig
plt.show()

## AP comparison


In [ ]:
fold_ap_df = pd.concat(
    {name: r.fold_ap for name, r in results.items()}, names=["augmentations"]
)
ap = fold_ap_df.groupby("augmentations", sort=False).mean()
ap.round(3)

In [ ]:
# difference to exp01 counts only if it has same sign on both folds
diff = (results["lecture"].fold_ap - results["current (exp01)"].fold_ap).T.add_prefix(
    "diff fold "
)
diff["same sign on both folds"] = (diff > 0).all(axis=1) | (diff < 0).all(axis=1)
diff.round(3)

## Grad-CAM


In [ ]:
cams = GradCamFigures(CONFIG)
for (cls, outcome), fig in cams.figures(VIP, N_SHOW).items():
    figures[f"exp03_lecture_augmentations_grad_cam_{cls.lower()}_{outcome}.png"] = fig
plt.show()

In [ ]:
with mlflow.start_run(
    experiment_id=client.get_experiment_by_name(CONFIG.experiment).experiment_id,
    run_name="summary",
):
    mlflow.log_param("val_mode", "fast")
    mlflow.log_table(fold_ap_df.reset_index(), "fold_ap.json")
    mlflow.log_table(diff.reset_index(names="metric"), "diff.json")
    mlflow.log_metrics({f"threshold_{c}": cams.thresholds[c] for c in VIP})
    if SAVE_FIGURES:
        for fname, fig in figures.items():
            fig.savefig(FIGURES / fname, dpi=120)
            mlflow.log_figure(fig, fname)